<a href="https://colab.research.google.com/github/polaya813/polaya813/blob/main/ETL_facts_P_Olaya.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Conjunto de datos simulados para el dim_products

In [6]:
import pandas as pd
import numpy as np

# Generar datos simulados para productos
np.random.seed(1)
num_products = 100
data_productos = {
    'producto_id_origen': np.arange(1, num_products + 1),
    'nombre_producto': [f'Producto {i}' for i in range(1, num_products + 1)],
    'categoria': np.random.choice(['Electrónica', 'Ropa', 'Hogar'], size=num_products),
    'precio': np.random.uniform(low=5, high=500, size=num_products)
}

# Crear un DataFrame
df_productos = pd.DataFrame(data_productos)

# Añadir una clave subrogada
df_productos['producto_key'] = range(1, len(df_productos) + 1)

# Guardar en un archivo CSV
df_productos.to_csv('dim_productos.csv', index=False)
print("Archivo dim_productos.csv generado con éxito.")


Archivo dim_productos.csv generado con éxito.


Conjunto simulado ventas_raw

In [7]:
import pandas as pd
import numpy as np

# Set a random seed for reproducibility
np.random.seed(42)

# Number of sales records to generate
num_sales = 1000

# Generate random sales data
data_ventas = {
    'venta_id': np.arange(1, num_sales + 1),
    'fecha': pd.date_range(start='2023-01-01', periods=num_sales, freq='H').strftime('%Y-%m-%d').tolist(),
    'ID de cliente': np.random.randint(1, 101, size=num_sales),
    'ID de producto': np.random.randint(1, 101, size=num_sales),
    'cantidad vendida': np.random.randint(1, 11, size=num_sales),
    'total venta': np.random.uniform(10, 1000, size=num_sales).round(2)
}

# Create a DataFrame
df_ventas = pd.DataFrame(data_ventas)

# Save to a CSV file
df_ventas.to_csv('ventas_raw.csv', index=False)
print("Archivo ventas_raw.csv generado con éxito.")


Archivo ventas_raw.csv generado con éxito.


/tmp/ipykernel_4535/3301824526.py:13: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  'fecha': pd.date_range(start='2023-01-01', periods=num_sales, freq='H').strftime('%Y-%m-%d').tolist(),


simular dim_tiempo

In [9]:
import pandas as pd

# Supongamos que las fechas en fact_ventas están en formato YYYYMMDD
# Generamos un rango de fechas basado en las fechas de fact_ventas
def simulate_dim_tiempo(fact_ventas_df):
    # Extraer las fechas únicas de fact_ventas
    unique_dates = pd.to_datetime(fact_ventas_df['fecha_key'], format='%Y%m%d').unique()

    # Crear un DataFrame para dim_tiempo
    dim_tiempo_df = pd.DataFrame({
        'fecha_key': unique_dates,
        'fecha_completa': unique_dates
    })

    # Convertir fecha_key a formato YYYYMMDD
    dim_tiempo_df['fecha_key'] = dim_tiempo_df['fecha_key'].dt.strftime('%Y%m%d').astype(int)

    # Convertir fecha_completa a formato legible
    dim_tiempo_df['fecha_completa'] = dim_tiempo_df['fecha_completa'].dt.strftime('%Y-%m-%d')

    return dim_tiempo_df

# Supongamos que ya tienes fact_ventas_df cargado
fact_ventas_df = pd.read_csv('fact_ventas.csv')

# Simular dim_tiempo
dim_tiempo_df = simulate_dim_tiempo(fact_ventas_df)

# Mostrar los primeros registros de dim_tiempo
print(dim_tiempo_df.head())

   fecha_key fecha_completa
0   20230101     2023-01-01
1   20230102     2023-01-02
2   20230103     2023-01-03
3   20230104     2023-01-04
4   20230105     2023-01-05


Asegúrate de tener los archivos dim_clientes.csv y dim_productos.csv disponibles en tu entorno. Estos archivos deben contener las claves subrogadas necesarias para el proceso de "lookup". Datos guardados en computador

In [13]:
import pandas as pd

# --- EXTRACT ---
def extract_ventas_raw(filepath='ventas_raw.csv'):
    """Extract raw sales data from a CSV file."""
    try:
        df_ventas_raw = pd.read_csv(filepath, parse_dates=['fecha'])
        print("Datos de ventas extraídos con éxito.")
        return df_ventas_raw
    except Exception as e:
        print(f"Error al extraer datos de ventas: {e}")
        return None

# --- TRANSFORM ---
def transform_f_ventas(df_ventas_raw, dim_clientes_df, dim_productos_df):
    """Transform raw sales data to match the fact table structure."""
    # Merge para obtener cliente_key
    df_transformed = pd.merge(df_ventas_raw,
                              dim_clientes_df[['cliente_id_origen', 'cliente_key']],
                              left_on='ID de cliente',
                              right_on='cliente_id_origen', how='left')

    # Merge para obtener producto_key
    df_transformed = pd.merge(df_transformed,
                              dim_productos_df[['producto_id_origen', 'producto_key']],
                              left_on='ID de producto',
                              right_on='producto_id_origen', how='left')

    # Generar fecha_key (ej. YYYYMMDD a partir de df_ventas_raw['fecha'])
    df_transformed['fecha_key'] = df_transformed['fecha'].dt.strftime('%Y%m%d').astype(int)

    # Renombrar columnas
    df_transformed.rename(columns={'cantidad vendida': 'cantidad_vendida', 'total venta': 'monto_total_venta'}, inplace=True)

    # Seleccionar y reordenar columnas para fact_ventas, incluyendo 'venta_id'
    fact_cols = ['venta_id', 'fecha_key', 'cliente_key', 'producto_key', 'cantidad_vendida', 'monto_total_venta']
    df_fact_ventas = df_transformed[fact_cols].dropna(subset=['cliente_key', 'producto_key', 'fecha_key'])

    print("Datos de ventas transformados con éxito.")
    return df_fact_ventas

# --- LOAD ---
def load_fact_ventas(df, output_filepath='fact_ventas.csv'):
    """Load transformed sales data into a CSV file."""
    try:
        df.to_csv(output_filepath, index=False)
        print(f"Datos cargados exitosamente en el archivo {output_filepath}.")
    except Exception as e:
        print(f"Error al cargar datos en el archivo: {e}")

# --- Ejecución del Pipeline para Hechos ---
def main():
    # Extraer datos de ventas
    df_ventas_raw = extract_ventas_raw()

    # Simular la carga de dimensiones (en un entorno real, esto vendría de tu DWH)
    dim_clientes_df = pd.read_csv('dim_clientes.csv')  # Asegúrate de tener este archivo
    dim_productos_df = pd.read_csv('dim_productos.csv')  # Asegúrate de tener este archivo

    if df_ventas_raw is not None:
        # Transformar datos de ventas
        df_fact_ventas = transform_f_ventas(df_ventas_raw, dim_clientes_df, dim_productos_df)
        # Cargar datos transformados
        load_fact_ventas(df_fact_ventas)

if __name__ == '__main__':
    main()

Datos de ventas extraídos con éxito.
Datos de ventas transformados con éxito.
Datos cargados exitosamente en el archivo fact_ventas.csv.


Para bajar archivo

In [ ]:
from google.colab import files
files.download('ventas_raw.csv')

Verificción de datos en el Dataware

In [14]:
import pandas as pd

# Cargar los datos desde los archivos CSV
fact_ventas_df = pd.read_csv('fact_ventas.csv')
dim_clientes_df = pd.read_csv('dim_clientes.csv')
dim_productos_df = pd.read_csv('dim_productos.csv')

# Simular la tabla dim_tiempo
# Asumimos que dim_tiempo tiene una columna 'fecha_key' y 'fecha_completa'
# Aseguramos que 'fecha_key' en dim_tiempo_df sea del mismo tipo (int64) que en fact_ventas_df
dim_tiempo_df = pd.DataFrame({
    'fecha_key': fact_ventas_df['fecha_key'], # Usar la fecha_key entera directamente de fact_ventas_df
    'fecha_completa': pd.to_datetime(fact_ventas_df['fecha_key'], format='%Y%m%d').dt.strftime('%Y-%m-%d')
})

# Realizar los joins para verificar los datos
result_df = fact_ventas_df.merge(dim_clientes_df, on='cliente_key', how='left') \
                          .merge(dim_productos_df, on='producto_key', how='left') \
                          .merge(dim_tiempo_df, on='fecha_key', how='left')

# Seleccionar las columnas necesarias
result_df = result_df[['venta_id', 'nombre_cliente', 'nombre_producto', 'fecha_completa', 'cantidad_vendida', 'monto_total_venta']]

# Mostrar los primeros 10 registros
print(result_df.head(10))

   venta_id nombre_cliente nombre_producto fecha_completa  cantidad_vendida  \
0         1     Cliente 52     Producto 34     2023-01-01                 9   
1         1     Cliente 52     Producto 34     2023-01-01                 9   
2         1     Cliente 52     Producto 34     2023-01-01                 9   
3         1     Cliente 52     Producto 34     2023-01-01                 9   
4         1     Cliente 52     Producto 34     2023-01-01                 9   
5         1     Cliente 52     Producto 34     2023-01-01                 9   
6         1     Cliente 52     Producto 34     2023-01-01                 9   
7         1     Cliente 52     Producto 34     2023-01-01                 9   
8         1     Cliente 52     Producto 34     2023-01-01                 9   
9         1     Cliente 52     Producto 34     2023-01-01                 9   

   monto_total_venta  
0             174.12  
1             174.12  
2             174.12  
3             174.12  
4             1

Generar un Diagrama ERD con ERAlchemy
ERAlchemy es una herramienta que puede generar diagramas ER a partir de un esquema de base de datos SQL. Si tienes un archivo de esquema SQL o una base de datos real, puedes usar ERAlchemy para crear un diagrama. Aquí te muestro cómo hacerlo si tuvieras un esquema SQL:

In [17]:
pip install eralchemy

Generar un Diagrama ERD:

Si tienes un archivo de esquema SQL, puedes usar el siguiente comando para generar un diagrama:

In [22]:
import pandas as pd

def generate_sql_schema(dataframes, primary_keys, foreign_keys, filename='schema.sql'):
    schema_lines = []
    for table_name, df in dataframes.items():
        schema_lines.append(f"CREATE TABLE {table_name} (")
        columns = []
        for col_name, dtype in df.dtypes.items():
            sql_type = "TEXT"
            if pd.api.types.is_integer_dtype(dtype):
                sql_type = "INTEGER"
            elif pd.api.types.is_float_dtype(dtype):
                sql_type = "REAL"
            elif pd.api.types.is_datetime64_any_dtype(dtype):
                sql_type = "DATETIME"

            col_definition = f"    {col_name} {sql_type}"
            if table_name in primary_keys and col_name == primary_keys[table_name]:
                col_definition += " PRIMARY KEY"
            columns.append(col_definition)

        # Add foreign key constraints
        if table_name in foreign_keys:
            for fk_col, fk_ref in foreign_keys[table_name].items():
                columns.append(f"    FOREIGN KEY ({fk_col}) REFERENCES {fk_ref['table']}({fk_ref['column']})")

        schema_lines.append(",\n".join(columns))
        schema_lines.append(");\n")

    with open(filename, 'w') as f:
        f.write("\n".join(schema_lines))
    print(f"Generated {filename} successfully.")

# Assuming these dataframes are available in your environment
dataframes_to_schema = {
    'fact_ventas': fact_ventas_df,
    'dim_clientes': dim_clientes_df,
    'dim_productos': dim_productos_df,
    'dim_tiempo': dim_tiempo_df
}

primary_keys_config = {
    'fact_ventas': 'venta_id',
    'dim_clientes': 'cliente_key',
    'dim_productos': 'producto_key',
    'dim_tiempo': 'fecha_key'
}

foreign_keys_config = {
    'fact_ventas': {
        'fecha_key': {'table': 'dim_tiempo', 'column': 'fecha_key'},
        'cliente_key': {'table': 'dim_clientes', 'column': 'cliente_key'},
        'producto_key': {'table': 'dim_productos', 'column': 'producto_key'}
    }
}

generate_sql_schema(dataframes_to_schema, primary_keys_config, foreign_keys_config)

# Now try to generate the ERD with eralchemy
!eralchemy -i schema.sql -o erd.pdf

print("Verifica el archivo erd.pdf en tu entorno.")

Generated schema.sql successfully.
rendering failed: Cannot process filename_or_input str: Could not parse SQLAlchemy URL from given URL string
Verifica el archivo erd.pdf en tu entorno.


Esto generará un archivo PDF con el diagrama ERD.

Generar un Diagrama ERD con pydot y Pandas
Si deseas crear un diagrama a partir de los DataFrames en Pandas, puedes usar pydot para crear un gráfico simple. Aquí te muestro un ejemplo básico:

In [23]:
import pandas as pd
import pydot

# Crear un gráfico dirigido
graph = pydot.Dot(graph_type='digraph')

# Añadir nodos para cada tabla
tables = ['fact_ventas', 'dim_clientes', 'dim_productos', 'dim_tiempo']
for table in tables:
    node = pydot.Node(table, shape='box')
    graph.add_node(node)

# Añadir aristas para las relaciones
edges = [
    ('fact_ventas', 'dim_clientes'),
    ('fact_ventas', 'dim_productos'),
    ('fact_ventas', 'dim_tiempo')
]

for edge in edges:
    graph.add_edge(pydot.Edge(edge[0], edge[1]))

# Guardar el gráfico en un archivo
graph.write_png('erd.png')
print("Diagrama ERD generado como erd.png.")

Diagrama ERD generado como erd.png.


Descripción del Código
Nodos: Cada tabla se representa como un nodo en el gráfico.
Aristas: Las relaciones entre las tablas se representan como aristas.
Salida: El gráfico se guarda como un archivo PNG.
Consideraciones
Limitaciones: Este enfoque es muy básico y no captura detalles como columnas o tipos de datos. Para diagramas más detallados, se recomienda usar herramientas especializadas como ERAlchemy con un esquema SQL real.
Visualización: Asegúrate de tener pydot y graphviz instalados para generar y visualizar el gráfico.
Este enfoque te permitirá crear un diagrama simple de las relaciones entre tus tablas simuladas. Si necesitas más ayuda o ajustes, no dudes en preguntar.